# ⚙️ Home Credit Default Risk - Data Preprocessing & Feature Engineering

## 📌 Mục Tiêu Xử Lý Dữ Liệu Cho Alternative Credit Scoring
Notebook này thực hiện các bước xử lý dữ liệu và tạo đặc trưng phái sinh (Feature Engineering) theo đúng chuẩn quy trình trong `AGENTS.md`:
1. **Xử lý giá trị bất thường (Outliers)**: Phát hiện và xử lý anomaly trong `DAYS_EMPLOYED` (`365243` -> `NaN` + flag `DAYS_EMPLOYED_ANOM`).
2. **Feature Engineering cho Alternative Data**: Tạo các chỉ số khả năng chi trả thay thế (`CREDIT_TO_INCOME_RATIO`, `ANNUITY_TO_INCOME_RATIO`, `PAYMENT_RATE`, `EMPLOYMENT_TO_AGE_RATIO`, `EXT_SOURCE_MEAN`, `EXT_SOURCE_MUL`).
3. **Tổng hợp dữ liệu đa bảng (Relational Aggregations)**: Gom nhóm thông tin lịch sử tín dụng từ `bureau.csv`, `previous_application.csv`, và `installments_payments.csv` theo `SK_ID_CURR`.
4. **Mã hóa biến định tính (Categorical Encoding)**: One-Hot Encoding cho các biến phân loại.
5. **Xuất bộ dữ liệu sạch (Processed Dataset)**: Lưu kết quả đã xử lý sẵn sàng cho bước huấn luyện mô hình.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
import time

DATA_DIR = Path('../data/raw/home-credit-default-risk')
if not DATA_DIR.exists():
    DATA_DIR = Path('data/raw/home-credit-default-risk')

OUTPUT_DIR = Path('../data/processed')
if not OUTPUT_DIR.parent.exists():
    OUTPUT_DIR = Path('data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✓ Đã chuẩn bị thư viện & đường dẫn lưu trữ dữ liệu.')

---
## 1. 📂 Processing Bảng Chính `application_train.csv`

In [ ]:
start_time = time.time()
app_df = pd.read_csv(DATA_DIR / 'application_train.csv')
print(f'► Shape ban đầu của application_train: {app_df.shape}')

# 1. Xử lý giá trị bất thường DAYS_EMPLOYED == 365243 (~1000 năm)
app_df['DAYS_EMPLOYED_ANOM'] = (app_df['DAYS_EMPLOYED'] == 365243).astype(int)
app_df['DAYS_EMPLOYED'] = app_df['DAYS_EMPLOYED'].replace(365243, np.nan)

# 2. Feature Engineering cho Alternative Credit Scoring
app_df['CREDIT_TO_INCOME_RATIO'] = app_df['AMT_CREDIT'] / (app_df['AMT_INCOME_TOTAL'] + 1)
app_df['ANNUITY_TO_INCOME_RATIO'] = app_df['AMT_ANNUITY'] / (app_df['AMT_INCOME_TOTAL'] + 1)
app_df['PAYMENT_RATE'] = app_df['AMT_ANNUITY'] / (app_df['AMT_CREDIT'] + 1)
app_df['EMPLOYMENT_TO_AGE_RATIO'] = np.abs(app_df['DAYS_EMPLOYED']) / (np.abs(app_df['DAYS_BIRTH']) + 1)
app_df['PHONE_CHANGE_YEARS'] = np.abs(app_df['DAYS_LAST_PHONE_CHANGE']) / 365.25

# Điểm tổng hợp từ bên thứ ba (External Alternative Scores)
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
app_df['EXT_SOURCE_MEAN'] = app_df[ext_cols].mean(axis=1)
app_df['EXT_SOURCE_STD'] = app_df[ext_cols].std(axis=1)
app_df['EXT_SOURCE_MUL'] = app_df['EXT_SOURCE_1'] * app_df['EXT_SOURCE_2'] * app_df['EXT_SOURCE_3']

print(f'✓ Hoàn tất feature engineering cho bảng chính ({time.time() - start_time:.2f}s)')

---
## 2. 🗄️ Gom Nhóm & Tổng Hợp Dữ Liệu Phụ (`bureau.csv`, `previous_application.csv`, `installments_payments.csv`)

In [ ]:
# Gom nhóm bureau.csv
bureau_path = DATA_DIR / 'bureau.csv'
if bureau_path.exists():
    print('► Đang xử lý bureau.csv...')
    bureau = pd.read_csv(bureau_path)
    bureau_agg = bureau.groupby('SK_ID_CURR').agg({
        'SK_ID_BUREAU': 'count',
        'DAYS_CREDIT': ['min', 'max', 'mean'],
        'CREDIT_DAY_OVERDUE': ['max', 'mean'],
        'AMT_CREDIT_SUM': ['sum', 'mean'],
        'AMT_CREDIT_SUM_DEBT': ['sum', 'mean']
    })
    bureau_agg.columns = ['BUREAU_' + '_'.join(col).upper() for col in bureau_agg.columns]
    app_df = app_df.merge(bureau_agg, on='SK_ID_CURR', how='left')
    print(f'✓ merged bureau_agg -> total cols: {app_df.shape[1]}')

# Gom nhóm previous_application.csv
prev_path = DATA_DIR / 'previous_application.csv'
if prev_path.exists():
    print('► Đang xử lý previous_application.csv...')
    prev = pd.read_csv(prev_path)
    prev_agg = prev.groupby('SK_ID_CURR').agg({
        'SK_ID_PREV': 'count',
        'AMT_APPLICATION': ['mean', 'max'],
        'AMT_CREDIT': ['mean', 'sum'],
        'AMT_ANNUITY': ['mean']
    })
    prev_agg.columns = ['PREV_' + '_'.join(col).upper() for col in prev_agg.columns]
    app_df = app_df.merge(prev_agg, on='SK_ID_CURR', how='left')
    print(f'✓ merged prev_agg -> total cols: {app_df.shape[1]}')

# Gom nhóm installments_payments.csv
inst_path = DATA_DIR / 'installments_payments.csv'
if inst_path.exists():
    print('► Đang xử lý installments_payments.csv...')
    inst = pd.read_csv(inst_path, nrows=2000000)
    inst['PAYMENT_PERC'] = inst['AMT_PAYMENT'] / (inst['AMT_INSTALMENT'] + 1)
    inst['PAYMENT_DIFF'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
    inst['DPD'] = (inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']).clip(lower=0)
    inst_agg = inst.groupby('SK_ID_CURR').agg({
        'DPD': ['max', 'mean'],
        'PAYMENT_PERC': ['mean'],
        'PAYMENT_DIFF': ['mean', 'sum']
    })
    inst_agg.columns = ['INST_' + '_'.join(col).upper() for col in inst_agg.columns]
    app_df = app_df.merge(inst_agg, on='SK_ID_CURR', how='left')
    print(f'✓ merged inst_agg -> total cols: {app_df.shape[1]}')

---
## 3. 🏷️ Categorical Encoding & Save Clean Data

In [ ]:
# One-Hot Encoding cho các biến object
cat_cols = app_df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'► Số lượng biến phân loại categorical: {len(cat_cols)}')

app_encoded = pd.get_dummies(app_df, columns=cat_cols, dummy_na=True, drop_first=True)
print(f'✓ Shape sau khi One-Hot Encoding: {app_encoded.shape}')

# Lưu dữ liệu đã xử lý ra data/processed/
output_file = OUTPUT_DIR / 'home_credit_processed.csv'
app_encoded.to_csv(output_file, index=False)
print(f'🎉 Đã lưu thành công bộ dữ liệu xử lý tại: {output_file}')
print(f'► Kích thước file: {output_file.stat().st_size / (1024*1024):.2f} MB')